[Open Google Colab](https://colab.research.google.com/) and choose **File → Upload notebook** to open this downloaded file. You can also run it in Jupyter or VS Code with a Python kernel. No GPU is needed.

# Session 5 lab: advanced search and profile methods

Work through BLAST's seed-and-extend heuristic, a position-specific scoring matrix and a profile Hidden Markov Model. Use the lecture's sentence and protein-alignment examples. For each hand calculation, write the working first, enter your result in place of None, then run the code check. You may use a calculator for logarithms. The larger computations are supplied so you can focus on the model and its interpretation.

By the end you should be able to calculate a word score, identify an X-drop endpoint, explain database-size effects on E-values, build selected PSSM entries, trace a sequence through a profile HMM, estimate transition/emission probabilities and interpret a real family match.

Answer the matching ten-question Canvas quiz as you work; every question is worth one point. Keep your calculations and brief explanations in this notebook. Use the Canvas module for the deadline and course-wide lab policy.

## Setup — first ten minutes
Run the next cell once. Then predict: what extra family information can a profile use that a single query sequence cannot? Write your initial answer in a Markdown cell. Revisit it after Part 3.

In [ ]:
import sys, importlib.util, subprocess
# These packages are normally available in Colab. Install only missing packages.
for module, package in [('numpy','numpy'), ('pandas','pandas'), ('matplotlib','matplotlib')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

CHECK_LOG=[]
def has_missing(value):
    if value is None: return True
    if isinstance(value,dict): return any(has_missing(v) for v in value.values())
    if isinstance(value,(list,tuple)): return any(has_missing(v) for v in value)
    return False

def answers_match(entered,expected,tol):
    if isinstance(expected,dict):
        return isinstance(entered,dict) and set(entered)==set(expected) and all(
            answers_match(entered[k],v,tol) for k,v in expected.items())
    if isinstance(expected,(list,tuple)):
        return isinstance(entered,(list,tuple)) and len(entered)==len(expected) and all(
            answers_match(a,b,tol) for a,b in zip(entered,expected))
    if isinstance(expected,(bool,np.bool_)):
        return isinstance(entered,(bool,np.bool_)) and bool(entered)==bool(expected)
    if isinstance(expected,(int,float,np.integer,np.floating)):
        return isinstance(entered,(int,float,np.integer,np.floating)) and not isinstance(entered,bool) and math.isclose(
            float(entered),float(expected),rel_tol=0,abs_tol=tol)
    if isinstance(expected,str):
        return isinstance(entered,str) and ' '.join(entered.split())==' '.join(expected.split())
    return entered==expected

def check(label,entered,expected,tol=.02):
    if has_missing(entered):
        status='pending'
        print(label+': enter your hand calculation in place of None, then rerun this cell.')
    elif answers_match(entered,expected,tol):
        status='correct'
        print(label+': correct.')
    else:
        status='retry'
        print(label+': check your working, the symbol order and the stated denominator, then try again.')
    CHECK_LOG.append({'label':label,'status':status})
    return None

plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
print('Setup complete. The required exercises use embedded data and do not submit remote jobs.')

## Part 1 — BLAST: seeds, extension and significance
**Slides 6–32**

BLAST saves time by extending promising short matches rather than aligning every sequence exhaustively. In this part you will calculate the word scores, follow an extension and interpret its significance.

### 1A. Neighbour words — 10 minutes
Use the query word **LWG** and the BLOSUM62 entries below. A word score is the sum of three substitution scores, one for each aligned position. A neighbour passes when its score is **at least T**.

| Query residue | Against L | Against I | Against W | Against G | Against A |
|---|---:|---:|---:|---:|---:|
| L | 4 | 2 | −2 | −4 | −1 |
| W | −2 | −3 | 11 | −2 | −3 |
| G | −4 | −4 | −2 | 6 | 0 |

Before running the checks:

1. Write the three terms and total for LWG against each of **LWG, IWG, LWA and AAA**.
2. Predict which pass T = 18.
3. Predict whether lowering T to 12 will increase or decrease the number of accepted words.

**Canvas Q1:** record your score for LWG against IWG.

In [ ]:
# Enter totals calculated on paper. Replace each None with a number.
my_word_scores = {'LWG': None, 'IWG': None, 'LWA': None, 'AAA': None}
my_words_at_18 = None  # a list of accepted words, e.g. ['...']

In [ ]:
BLOSUM62 = {'A': {'A': 4, 'R': -1, 'N': -2, 'D': -2, 'C': 0, 'Q': -1, 'E': -1, 'G': 0, 'H': -2, 'I': -1, 'L': -1, 'K': -1, 'M': -1, 'F': -2, 'P': -1, 'S': 1, 'T': 0, 'W': -3, 'Y': -2, 'V': 0}, 'R': {'A': -1, 'R': 5, 'N': 0, 'D': -2, 'C': -3, 'Q': 1, 'E': 0, 'G': -2, 'H': 0, 'I': -3, 'L': -2, 'K': 2, 'M': -1, 'F': -3, 'P': -2, 'S': -1, 'T': -1, 'W': -3, 'Y': -2, 'V': -3}, 'N': {'A': -2, 'R': 0, 'N': 6, 'D': 1, 'C': -3, 'Q': 0, 'E': 0, 'G': 0, 'H': 1, 'I': -3, 'L': -3, 'K': 0, 'M': -2, 'F': -3, 'P': -2, 'S': 1, 'T': 0, 'W': -4, 'Y': -2, 'V': -3}, 'D': {'A': -2, 'R': -2, 'N': 1, 'D': 6, 'C': -3, 'Q': 0, 'E': 2, 'G': -1, 'H': -1, 'I': -3, 'L': -4, 'K': -1, 'M': -3, 'F': -3, 'P': -1, 'S': 0, 'T': -1, 'W': -4, 'Y': -3, 'V': -3}, 'C': {'A': 0, 'R': -3, 'N': -3, 'D': -3, 'C': 9, 'Q': -3, 'E': -4, 'G': -3, 'H': -3, 'I': -1, 'L': -1, 'K': -3, 'M': -1, 'F': -2, 'P': -3, 'S': -1, 'T': -1, 'W': -2, 'Y': -2, 'V': -1}, 'Q': {'A': -1, 'R': 1, 'N': 0, 'D': 0, 'C': -3, 'Q': 5, 'E': 2, 'G': -2, 'H': 0, 'I': -3, 'L': -2, 'K': 1, 'M': 0, 'F': -3, 'P': -1, 'S': 0, 'T': -1, 'W': -2, 'Y': -1, 'V': -2}, 'E': {'A': -1, 'R': 0, 'N': 0, 'D': 2, 'C': -4, 'Q': 2, 'E': 5, 'G': -2, 'H': 0, 'I': -3, 'L': -3, 'K': 1, 'M': -2, 'F': -3, 'P': -1, 'S': 0, 'T': -1, 'W': -3, 'Y': -2, 'V': -2}, 'G': {'A': 0, 'R': -2, 'N': 0, 'D': -1, 'C': -3, 'Q': -2, 'E': -2, 'G': 6, 'H': -2, 'I': -4, 'L': -4, 'K': -2, 'M': -3, 'F': -3, 'P': -2, 'S': 0, 'T': -2, 'W': -2, 'Y': -3, 'V': -3}, 'H': {'A': -2, 'R': 0, 'N': 1, 'D': -1, 'C': -3, 'Q': 0, 'E': 0, 'G': -2, 'H': 8, 'I': -3, 'L': -3, 'K': -1, 'M': -2, 'F': -1, 'P': -2, 'S': -1, 'T': -2, 'W': -2, 'Y': 2, 'V': -3}, 'I': {'A': -1, 'R': -3, 'N': -3, 'D': -3, 'C': -1, 'Q': -3, 'E': -3, 'G': -4, 'H': -3, 'I': 4, 'L': 2, 'K': -3, 'M': 1, 'F': 0, 'P': -3, 'S': -2, 'T': -1, 'W': -3, 'Y': -1, 'V': 3}, 'L': {'A': -1, 'R': -2, 'N': -3, 'D': -4, 'C': -1, 'Q': -2, 'E': -3, 'G': -4, 'H': -3, 'I': 2, 'L': 4, 'K': -2, 'M': 2, 'F': 0, 'P': -3, 'S': -2, 'T': -1, 'W': -2, 'Y': -1, 'V': 1}, 'K': {'A': -1, 'R': 2, 'N': 0, 'D': -1, 'C': -3, 'Q': 1, 'E': 1, 'G': -2, 'H': -1, 'I': -3, 'L': -2, 'K': 5, 'M': -1, 'F': -3, 'P': -1, 'S': 0, 'T': -1, 'W': -3, 'Y': -2, 'V': -2}, 'M': {'A': -1, 'R': -1, 'N': -2, 'D': -3, 'C': -1, 'Q': 0, 'E': -2, 'G': -3, 'H': -2, 'I': 1, 'L': 2, 'K': -1, 'M': 5, 'F': 0, 'P': -2, 'S': -1, 'T': -1, 'W': -1, 'Y': -1, 'V': 1}, 'F': {'A': -2, 'R': -3, 'N': -3, 'D': -3, 'C': -2, 'Q': -3, 'E': -3, 'G': -3, 'H': -1, 'I': 0, 'L': 0, 'K': -3, 'M': 0, 'F': 6, 'P': -4, 'S': -2, 'T': -2, 'W': 1, 'Y': 3, 'V': -1}, 'P': {'A': -1, 'R': -2, 'N': -2, 'D': -1, 'C': -3, 'Q': -1, 'E': -1, 'G': -2, 'H': -2, 'I': -3, 'L': -3, 'K': -1, 'M': -2, 'F': -4, 'P': 7, 'S': -1, 'T': -1, 'W': -4, 'Y': -3, 'V': -2}, 'S': {'A': 1, 'R': -1, 'N': 1, 'D': 0, 'C': -1, 'Q': 0, 'E': 0, 'G': 0, 'H': -1, 'I': -2, 'L': -2, 'K': 0, 'M': -1, 'F': -2, 'P': -1, 'S': 4, 'T': 1, 'W': -3, 'Y': -2, 'V': -2}, 'T': {'A': 0, 'R': -1, 'N': 0, 'D': -1, 'C': -1, 'Q': -1, 'E': -1, 'G': -2, 'H': -2, 'I': -1, 'L': -1, 'K': -1, 'M': -1, 'F': -2, 'P': -1, 'S': 1, 'T': 5, 'W': -2, 'Y': -2, 'V': 0}, 'W': {'A': -3, 'R': -3, 'N': -4, 'D': -4, 'C': -2, 'Q': -2, 'E': -3, 'G': -2, 'H': -2, 'I': -3, 'L': -2, 'K': -3, 'M': -1, 'F': 1, 'P': -4, 'S': -3, 'T': -2, 'W': 11, 'Y': 2, 'V': -3}, 'Y': {'A': -2, 'R': -2, 'N': -2, 'D': -3, 'C': -2, 'Q': -1, 'E': -2, 'G': -3, 'H': 2, 'I': -1, 'L': -1, 'K': -2, 'M': -1, 'F': 3, 'P': -3, 'S': -2, 'T': -2, 'W': 2, 'Y': 7, 'V': -1}, 'V': {'A': 0, 'R': -3, 'N': -3, 'D': -3, 'C': -1, 'Q': -2, 'E': -2, 'G': -3, 'H': -3, 'I': 3, 'L': 1, 'K': -2, 'M': 1, 'F': -1, 'P': -2, 'S': -2, 'T': 0, 'W': -3, 'Y': -1, 'V': 4}}

AA_BLAST = 'ARNDCQEGHILKMFPSTWYV'
def word_score(query, word):
    if len(query) != len(word):
        raise ValueError('Words must have the same length.')
    return sum(BLOSUM62[a][b] for a, b in zip(query, word))

expected_word_scores = {w: word_score('LWG', w) for w in my_word_scores}
check('Word scores', my_word_scores, expected_word_scores, tol=0)
check('Words passing T=18', sorted(my_words_at_18) if my_words_at_18 is not None else None,
      sorted(w for w, s in expected_word_scores.items() if s >= 18), tol=0)

### Test the threshold prediction
The next cell checks all 8,000 three-letter protein words. Run it first with T = 18, then change T to 12 and to 22. Explain why the workload changes. A neighbour is a potential seed, not evidence of homology by itself.

In [ ]:
from itertools import product
T = 18
all_words = [''.join(w) for w in product(AA_BLAST, repeat=3)]
neighbours = sorted(((w, word_score('LWG', w)) for w in all_words
                     if word_score('LWG', w) >= T), key=lambda x: (-x[1], x[0]))
print(f'{len(neighbours)} words pass T={T}. First 12:')
print(neighbours[:12])

### 1B. Extension and X-drop — 15 minutes
Use the sentence example from slides 14–23:

    query:   The quick brown fox jumps over the lazy dog.
    target:  The quiet brown cat purrs when she sees him.

Ignore spaces and punctuation. Start at the left-hand T. Score **+1 for an identical character** and **−1 otherwise**, with no gaps. Start the running score and best score at zero. At each column update the running score, then the best score. Stop at the **first column where best score − running score is at least X**, using X = 5. Return the alignment only up to the best-scoring endpoint. This explicit boundary rule is used throughout this lab.

Make a paper table with columns: position, query character, target character, running score, best score and drop. Complete positions 1–20. Mark the best endpoint and the stopping position; they need not be the same.

**Canvas Q2:** record the peak score, the reported alignment length, and the stopping position.

In [ ]:
my_xdrop = {'peak_score': None, 'reported_length': None, 'stop_position': None}
# Complete these selected rows from your paper trace.
my_running_scores = {6: None, 8: None, 13: None, 18: None, 20: None}

In [ ]:
QUERY_TEXT = 'The quick brown fox jumps over the lazy dog.'
TARGET_TEXT = 'The quiet brown cat purrs when she sees him.'
def xdrop_trace(query, target, X=5):
    if X <= 0:
        raise ValueError('Choose a positive X-drop threshold.')
    query = ''.join(c for c in query if c.isalpha())
    target = ''.join(c for c in target if c.isalpha())
    score, best, endpoint = 0, 0, 0
    trace, stop = [], None
    for position, (a, b) in enumerate(zip(query, target), start=1):
        score += 1 if a == b else -1
        if score > best:
            best, endpoint = score, position
        drop = best - score
        trace.append(dict(position=position, query=a, target=b,
                          running=score, best=best, drop=drop))
        if drop >= X:
            stop = position
            break
    return dict(trace=trace, peak_score=best, reported_length=endpoint,
                stop_position=stop, aligned_query=query[:endpoint],
                aligned_target=target[:endpoint])

extension = xdrop_trace(QUERY_TEXT, TARGET_TEXT, X=5)
check('Peak, reported length and stopping point', my_xdrop,
      {k: extension[k] for k in my_xdrop}, tol=0)
expected_rows = {r['position']: r['running'] for r in extension['trace']
                 if r['position'] in my_running_scores}
check('Selected running scores', my_running_scores, expected_rows, tol=0)

After checking your hand trace, inspect the full trace and plot below. Why does the reported alignment end before extension stops? Predict what will happen if X is changed from 5 to 3 or 7, then change X in the cell and inspect the result. Explain the extra sequence comparisons allowed by a larger X.

In [ ]:
X = 5
plotted = xdrop_trace(QUERY_TEXT, TARGET_TEXT, X=X)
display(pd.DataFrame(plotted['trace']))
print('Reported query :', plotted['aligned_query'])
print('Reported target:', plotted['aligned_target'])
positions = [r['position'] for r in plotted['trace']]
running = [r['running'] for r in plotted['trace']]
peaks = [r['best'] for r in plotted['trace']]
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(positions, running, 'o-', label='Running score')
ax.plot(positions, peaks, '--', label='Best score so far')
ax.plot(positions, [p-X for p in peaks], ':', label=f'Best minus X ({X})')
if plotted['reported_length']:
    ax.axvline(plotted['reported_length'], color='#25824b', alpha=.6, label='Reported endpoint')
if plotted['stop_position'] is not None:
    ax.axvline(plotted['stop_position'], color='#b64343', alpha=.6, label='Extension stops')
ax.set(xlabel='Aligned character position', ylabel='Score', title='X-drop extension')
ax.legend(loc='best', fontsize=8); ax.grid(alpha=.15)
fig.tight_layout(); plt.show()

### 1C. Raw score, bit score and E-value — 15 minutes
For this numerical exercise use a simplified statistical model:

$$B = \frac{\lambda S-\ln K}{\ln 2},\qquad E=Kmn e^{-\lambda S}=mn2^{-B}.$$

Here S is the raw score, B the bit score, m the query length and n the total number of database residues. Real BLAST uses effective search lengths and additional corrections; the supplied numbers make the arithmetic transparent.

Use **S = 50, λ = ln 2, K = 1, m = 100 and n = 1,000,000**.

1. Calculate B and E with a calculator. Retain at least three significant figures.
2. Calculate E for a database ten times as large, keeping the same alignment and scoring system.
3. Predict whether B changes. Explain why an E-value is an expected chance-hit count, rather than the probability that the sequences are unrelated.

**Canvas Q3:** identify which quantities change when the database grows tenfold.

In [ ]:
my_statistics = {'bits': None, 'E_initial': None, 'E_tenfold': None, 'bits_tenfold': None}

In [ ]:
def bit_score(raw, lam, K):
    return (lam * raw - math.log(K)) / math.log(2)
def expected_hits(bits, query_length, database_residues):
    return query_length * database_residues * 2.0 ** (-bits)
bits = bit_score(50, math.log(2), 1)
E_initial = expected_hits(bits, 100, 1_000_000)
E_tenfold = expected_hits(bits, 100, 10_000_000)
check('Bit scores', {k: my_statistics[k] for k in ['bits', 'bits_tenfold']},
      {'bits': bits, 'bits_tenfold': bits}, tol=1e-6)
check('Initial E-value', my_statistics['E_initial'], E_initial, tol=E_initial*.02)
check('Tenfold-database E-value', my_statistics['E_tenfold'], E_tenfold, tol=E_tenfold*.02)
print('A tenfold increase in search space multiplies E by ten for a fixed bit score.')

## Part 2 — Build and use a position-specific scoring matrix

**Slides 33–53**

A family alignment gives more information than one query sequence. Use the same five short sequences as in the lecture, construct selected profile entries by hand, then check them with code. Keep your hand calculations for lab quiz Q4–Q6.

Work through four stages: column counts and frequencies (10 min), log-odds and candidate scores (15 min), gap placement (12 min), and iteration decisions (8 min). These are deliberately small teaching models. The additive pseudocount and 21-symbol gap example do not reproduce PSI-BLAST's complete profile-building procedure.

The full 20-residue background below is given for this lab. These rounded BLOSUM62 frequencies sum to one; they match the [HMMER/Easel background-frequency source](https://raw.githubusercontent.com/EddyRivasLab/easel/master/esl_composition.c), function `esl_composition_BL62`. The same values appear in [Altschul et al. (2009), Table 3](https://doi.org/10.1093/nar/gkn981). We use the alignment from the lecture and calculate all scores from the stated rules.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

MSA = np.array([list(x) for x in ["AGLSP", "AGLTP", "RGISP", "AALSQ", "AALSP"]])
BACKGROUND = {
    "A": .074, "R": .052, "N": .045, "D": .054, "C": .025,
    "Q": .034, "E": .054, "G": .074, "H": .026, "I": .068,
    "L": .099, "K": .058, "M": .025, "F": .047, "P": .039,
    "S": .057, "T": .051, "W": .013, "Y": .032, "V": .073,
}
assert len(BACKGROUND) == 20 and np.isclose(sum(BACKGROUND.values()), 1)
print("Five sequences, five aligned positions")
display(pd.DataFrame(MSA, index=range(1, 6), columns=range(1, 6)))
print("Given background frequencies q (all 20 residue types)")
display(pd.DataFrame([BACKGROUND], index=["q"]).round(3))

### Count columns 1 and 5 by hand

Count A, R and G in column 1. Count P, Q and G in column 5. A residue absent from a column has count zero; it still belongs to the 20-residue alphabet. Enter the counts below.

In [ ]:
# Count the two columns by hand before running this cell.
hand_counts = {
    "column_1": {"A": None, "R": None, "G": None},
    "column_5": {"P": None, "Q": None, "G": None},
}
check("Column counts", hand_counts, {
    "column_1": {a: MSA[:, 0].tolist().count(a) for a in "ARG"},
    "column_5": {a: MSA[:, 4].tolist().count(a) for a in "PQG"},
})

### Convert counts to frequencies

Add one pseudocount for **each of the 20 amino acids**. With five sequences, use

\[
f_{i,a} = \frac{c_{i,a}+1}{5+20}.
\]

Without pseudocounts, G at column 1 has frequency 0/5 and log-odds score \(-\infty\). A candidate with G at this position would therefore have total score \(-\infty\), regardless of its other residues. Explain why five observations are insufficient to justify that hard exclusion, and what adding +1 changes.

Calculate the denominator and the five selected frequencies below. Explain why using only the residue types observed in this alignment would describe a different model.

In [ ]:
# Add one count for EACH of the 20 amino acids, including unobserved ones.
hand_denominator = None
hand_frequencies = {
    "A_at_1": None, "R_at_1": None, "G_at_1": None,
    "P_at_5": None, "Q_at_5": None,
}
check("Frequency denominator", hand_denominator, len(MSA) + len(BACKGROUND))
check("Smoothed frequencies", hand_frequencies, {
    "A_at_1": (np.count_nonzero(MSA[:, 0] == "A") + 1) / 25,
    "R_at_1": (np.count_nonzero(MSA[:, 0] == "R") + 1) / 25,
    "G_at_1": (np.count_nonzero(MSA[:, 0] == "G") + 1) / 25,
    "P_at_5": (np.count_nonzero(MSA[:, 4] == "P") + 1) / 25,
    "Q_at_5": (np.count_nonzero(MSA[:, 4] == "Q") + 1) / 25,
})

In [ ]:
def make_profile(alignment, background):
    """Add one to EVERY symbol; rows are 1-based alignment positions."""
    symbols = list(background)
    counts = np.array([[np.count_nonzero(alignment[:, i] == a) for a in symbols]
                       for i in range(alignment.shape[1])])
    frequencies = (counts + 1) / (len(alignment) + len(symbols))
    assert np.allclose(frequencies.sum(axis=1), 1)
    scores = np.log2(frequencies / np.array([background[a] for a in symbols]))
    return pd.DataFrame(scores, index=range(1, alignment.shape[1] + 1), columns=symbols)


def score_profile(aligned_string, profile):
    if len(aligned_string) != len(profile):
        raise ValueError("One symbol is needed for every profile position.")
    return sum(float(profile.loc[i, a]) for i, a in enumerate(aligned_string, 1))


PSSM = make_profile(MSA, BACKGROUND)
# Print the given q values needed for the next hand calculation.
for a in "ARG LSP".replace(" ", ""):
    print(f"q({a}) = {BACKGROUND[a]:.3f}")

### Calculate and add log-odds scores

For residue \(a\) at position \(i\), use \(w_{i,a}=\log_2(f_{i,a}/q_a)\). A positive value favours that residue relative to its background frequency; a negative value means it is less frequent in this model than in the background. A smoothed unseen residue has a finite score.

1. Calculate A, R and G scores in column 1. The G value is needed for **Q4**.
2. Score **AGLSP** and **GGLSP** by calculating the five required log-odds entries from the alignment counts and given background frequencies, then adding them. Write each five-term sum before entering the total.
3. Calculate **S(AGLSP) − S(GGLSP)** for **Q5**. Explain why the two candidates differ only in the contribution from column 1. Because A and G have the same background frequency here, their background terms cancel in the difference.

These toy profile scores use base-2 log-odds. They are not calibrated BLAST bit scores or E-values.

In [ ]:
# Use log2(f / q). Round your entries to at least two decimal places.
hand_log_odds = {"A_at_1": None, "R_at_1": None, "G_at_1": None}
hand_sequence_scores = {"AGLSP": None, "GGLSP": None}
hand_difference = None
check("Position 1 log-odds", hand_log_odds,
      {a + "_at_1": float(PSSM.loc[1, a]) for a in "ARG"})
check("Candidate scores", hand_sequence_scores,
      {seq: score_profile(seq, PSSM) for seq in ["AGLSP", "GGLSP"]})
check("Candidate score difference", hand_difference,
      score_profile("AGLSP", PSSM) - score_profile("GGLSP", PSSM))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.8))
limit = float(np.max(np.abs(PSSM.to_numpy())))
im = ax.imshow(PSSM.to_numpy(), aspect="auto", cmap="RdBu_r", vmin=-limit, vmax=limit)
ax.set_xticks(range(len(PSSM.columns)), PSSM.columns)
ax.set_yticks(range(len(PSSM.index)), PSSM.index)
ax.set_xlabel("Candidate residue")
ax.set_ylabel("Alignment position (1-based)")
ax.set_title("Twenty-residue PSSM: log-odds after +1 pseudocounts")
fig.colorbar(im, ax=ax, label="Toy log-odds score (bits)")
fig.tight_layout()
plt.show()

frequency_consensus = "".join(max(BACKGROUND, key=lambda a: np.count_nonzero(MSA[:, i] == a))
                              for i in range(MSA.shape[1]))
log_odds_maximum = "".join(PSSM.loc[i].idxmax() for i in PSSM.index)
print("Most frequent residue in each column:", frequency_consensus)
print("Highest log-odds residue in each column:", log_odds_maximum)
print("These describe different criteria; they need not produce the same sequence.")

Compare the two sequences printed above. At column 1, inspect the score of W, which was never observed. Explain how a uniform +1 prior and a rare background residue can produce this result. Do not interpret the highest-scoring sequence as an experimentally established optimal protein. This is a model built from five short sequences with a deliberately simple prior.

### Compare gap placements with a stated toy rule

A PSSM stores residue scores. A real sequence-to-profile alignment normally uses a separate gap model. To see why placement matters, first treat **-** as a 21st symbol in the six-column alignment below. This is a transparent scoring exercise, not the PSI-BLAST gap algorithm.

There are 30 symbols in this alignment. Calculate the total number of gaps. Set \(q_{-}=\text{number of gaps}/30\), and rescale each amino-acid background to \((1-q_{-})q_a\), so all 21 background probabilities sum to one. Add +1 to every symbol count, including the gap, giving denominator \(5+21\).

The candidate **RASRP** has five residues and needs one gap to span the six columns. Compare **-RASRP** with **RA-SRP**. Predict which better follows the alignment's column preferences. Use the supplied score table to add all six position-specific entries for each candidate **by hand**, then enter both totals. Keep each written sum. After checking the two sums, enumerate all six placements.

In [ ]:
GAPPED_MSA = np.array([list(x) for x in ["AGLS-P", "AGLTRP", "RGISRP", "AALS-Q", "AA-S-P"]])
display(pd.DataFrame(GAPPED_MSA, index=range(1, 6), columns=range(1, 7)))
q_gap = float(np.count_nonzero(GAPPED_MSA == "-") / GAPPED_MSA.size)
GAP_BACKGROUND = {a: (1 - q_gap) * q for a, q in BACKGROUND.items()}
GAP_BACKGROUND["-"] = q_gap
assert np.isclose(sum(GAP_BACKGROUND.values()), 1)
GAP_PSSM = make_profile(GAPPED_MSA, GAP_BACKGROUND)
print("Given score table for hand addition: choose one entry per position")
display(GAP_PSSM.loc[:, ["A", "R", "S", "P", "-"]].round(3))
# Use the displayed table to add the two candidate scores before continuing.

In [ ]:
# Predict the stronger placement before revealing all six scores.
# Enter the aligned string as text, including its one gap.
hand_better_placement = None
hand_gap_count = None
hand_gap_denominator = None
# Add six supplied table entries for EACH aligned candidate by hand.
hand_gap_scores = {"-RASRP": None, "RA-SRP": None}
check("Gap count in the whole alignment", hand_gap_count,
      int(np.count_nonzero(GAPPED_MSA == "-")))
check("Twenty-one-symbol denominator", hand_gap_denominator,
      len(GAPPED_MSA) + len(GAP_BACKGROUND))
check("Stronger of the two proposed placements", hand_better_placement,
      max(["-RASRP", "RA-SRP"], key=lambda x: score_profile(x, GAP_PSSM)))
check("Two hand-scored gap placements", hand_gap_scores,
      {x: score_profile(x, GAP_PSSM) for x in ["-RASRP", "RA-SRP"]})

In [ ]:
candidate = "RASRP"
placements = [candidate[:i] + "-" + candidate[i:] for i in range(len(candidate) + 1)]
placement_scores = pd.DataFrame({
    "aligned candidate": placements,
    "gap column": [x.index("-") + 1 for x in placements],
    "score": [score_profile(x, GAP_PSSM) for x in placements],
}).sort_values("score", ascending=False).reset_index(drop=True)
display(placement_scores.round({"score": 3}))
print("Each row scores the same five residues and one gap against the same model.")

Record the strongest placement and its gap column for **Q6**. Explain the ranking using the observed alignment columns. A position-specific preference can favour one gap location over another. A profile HMM represents insertion and deletion behaviour through states and transition probabilities, rather than treating a gap as an emitted 21st residue.

In [ ]:
# Record the highest-scoring placement and its 1-based gap column.
hand_best_gap_placement = None
hand_best_gap_column = None
best_gap_string = max(placements, key=lambda x: score_profile(x, GAP_PSSM))
check("Best placement", hand_best_gap_placement, best_gap_string)
check("Best gap column", hand_best_gap_column, best_gap_string.index("-") + 1)

### Decide which hits should enter the next profile

This small table is a **teaching scenario**, not output from a live search. Suppose the reporting threshold is E ≤ 0.01 and the inclusion threshold is E ≤ 0.001. Only hits spanning at least 80% of the intended family region may enter this exercise's next profile.

| Hit | E-value | Coverage of intended region | Inspection |
| --- | ---: | ---: | --- |
| A | 1e-12 | 98% | Expected family motifs |
| B | 4e-4 | 91% | Expected family motifs |
| C | 5e-4 | 18% | Match confined to a low-complexity segment |
| D | 5e-3 | 96% | Borderline reported hit |

All four are reported. Decide which should enter the next profile under the stated rules. Then predict what might happen in round 3 if C were included because someone considered its E-value alone. Identify one check you would make before accepting an unexpected late-round hit.

In [ ]:
# Use the thresholds and coverage evidence in the table above.
# Enter a list of accession labels, for example ["A", "C"].
hand_include_round_2 = None
check("Round 2 profile entries", hand_include_round_2, ["A", "B"])
# In your notes explain why reporting a hit and including it are different decisions.
# Predict what could happen in round 3 if low-complexity hit C were included anyway.

Before moving on, retain your hand counts, two candidate sums and explanation of the best gap placement. The general PSI-BLAST cycle is **search → select appropriate included matches → build a weighted profile → search again**. Reporting a hit does not automatically make it suitable training data for the next profile. A late-round hit needs inspection, because an unsuitable included match can pull subsequent rounds towards unrelated sequences.

## Break

Take a break. When you return, keep the distinction between an alignment column and a sequence position in mind: insertions and deletions mean they need not have the same numbering.

## Part 3. Build and use a profile HMM (1:45–2:35; 50 minutes)

Use the lecture's alignment (slides 54–121) to choose model columns, trace sequences through states, estimate probabilities and infer a new sequence's path.

This is a **generic toy profile HMM** with all nine interior transition types: MM, MI, MD, IM, II, ID, DM, DI and DD. In particular, D5→I5 is allowed. These declared rules define our teaching model, rather than HMMER's specific architecture or training procedure.

Match and insert states emit residues. Delete states are **silent**: they advance through the model without consuming a query residue. Dots used to pad insertion regions are neither residues nor delete states.

### 3A. Alignment columns and paths (10 minutes)

    Column       1 2 3 4 5 6 7 8 9 10 11
    Sequence 1   M A K L T - - - K  -  D
    Sequence 2   M A K L T - - - R  P  -
    Sequence 3   - G K S R - - - R  T  E
    Sequence 4   M A R S - Q N A H  Y  N
    Sequence 5   M A R S R - - - H  W  Q

Treat columns with residues in at least half the sequences as match columns. The three neighbouring sparse columns form one insertion region, called I5 because it follows the fifth match column. Number match columns M1–M8 from left to right.

Identify the original match and insertion column numbers. Trace sequences 1 and 4 from B to E, including delete states and omitting insertion-padding gaps. Enter paths as space-separated strings, for example B M1 D2 M3 E.

In [ ]:
from matplotlib.patches import FancyArrowPatch
fig, ax = plt.subplots(figsize=(9, 4))
positions = {'Mi': (0, 2), 'Di': (0, 1), 'Ii': (1, 0),
             'Mn': (4, 2), 'Dn': (4, 1)}
labels = {'Mi': r'$M_i$', 'Di': r'$D_i$', 'Ii': r'$I_i$',
          'Mn': r'$M_{i+1}$', 'Dn': r'$D_{i+1}$'}
# Nine declared interior transition types, including the insert self-loop.
edges = [('Mi','Mn',0), ('Mi','Dn',-.12), ('Mi','Ii',.12),
         ('Di','Mn',.12), ('Di','Dn',0), ('Di','Ii',-.1),
         ('Ii','Mn',-.12), ('Ii','Dn',.12)]
for start, end, curve in edges:
    ax.add_patch(FancyArrowPatch(positions[start], positions[end],
        connectionstyle=f'arc3,rad={curve}', arrowstyle='-|>',
        mutation_scale=14, shrinkA=22, shrinkB=22, color='#617b8a', linewidth=1.35))
ax.add_patch(FancyArrowPatch((.82,-.02),(1.18,-.02),
    connectionstyle='arc3,rad=2.2',arrowstyle='-|>',mutation_scale=14,
    color='#617b8a',linewidth=1.35))
for state,(x,y) in positions.items():
    silent = state.startswith('D')
    ax.text(x,y,labels[state],ha='center',va='center',fontsize=14,
        bbox=dict(boxstyle='round,pad=.48',facecolor='#f3dfb4' if silent else '#dceff0',
                  edgecolor='#5c727c'))
ax.text(2.7,-.68,'I self-loop: emit another residue',ha='center',fontsize=9)
ax.text(2.15,2.55,'Repeating interior unit of the stated toy model',ha='center',fontsize=12)
ax.text(2.1,-1.0,'M and I emit residues; D is silent. Boundary edges are listed in Part 3B.',
        ha='center',fontsize=9)
ax.set(xlim=(-.6,4.8),ylim=(-1.2,2.85));ax.axis('off')
fig.tight_layout();plt.show()

In [ ]:
hmm_structure = {
 "match_columns": None,       # tuple of original MSA column numbers
 "insert_columns": None,      # tuple of original MSA column numbers
 "sequence_1_path": None,     # string including B and E
 "sequence_4_path": None,
 "delete_emits_residue": None,# True or False
}

In [ ]:
check("HMM columns and paths",hmm_structure,{
 "match_columns": (1,2,3,4,5,9,10,11), "insert_columns": (6,7,8),
 "sequence_1_path": "B M1 M2 M3 M4 M5 M6 D7 M8 E",
 "sequence_4_path": "B M1 M2 M3 M4 D5 I5 I5 I5 M6 M7 M8 E",
 "delete_emits_residue": False,
})

### 3B. Transition probabilities (10 minutes)

Count **outgoing transitions from a state**, then divide by their total. A run of three residues in I5 contains two I5→I5 transitions followed by an exit. The transition entering I5 belongs to the previous state's distribution.

1. Count I5→I5, I5→M6 and I5→D6 over the five paths. Calculate their raw probabilities. Keep the unsmoothed I5 self-loop probability for **Q7**.
2. Count M1→M2, M1→D2 and M1→I1. Add +1 to each of those three edge counts and normalise.
3. Apply the same +1 rule to I5's outgoing edges. Compare its smoothed and raw distributions.

For the full model, B and I0 can lead to M1, D1 or I0. At interior column i, each of M_i, D_i and I_i can lead to M_(i+1), D_(i+1) or I_i. At the final column, M8, D8 and I8 can lead to E or I8. No other transitions are allowed. An unobserved transition row becomes uniform over its declared edges after smoothing.

In [ ]:
hmm_transition_answers = {
 "I5_counts_II_IM_ID": None,                 # tuples in the stated order
 "I5_raw_probabilities_II_IM_ID": None,
 "M1_counts_MM_MD_MI": None,
 "M1_plus1_probabilities_MM_MD_MI": None,
 "I5_plus1_probabilities_II_IM_ID": None,
}

In [ ]:
check("HMM transitions",hmm_transition_answers,{
 "I5_counts_II_IM_ID": (2,1,0), "I5_raw_probabilities_II_IM_ID": (2/3,1/3,0),
 "M1_counts_MM_MD_MI": (4,0,0), "M1_plus1_probabilities_MM_MD_MI": (5/7,1/7,1/7),
 "I5_plus1_probabilities_II_IM_ID": (3/6,2/6,1/6),
},tol=0.005)

### 3C. Emission probabilities (10 minutes)

M6 corresponds to **original alignment column 9**, not column 6. Count its K, R and H observations. Separately pool all residues emitted by I5.

Calculate the raw emission probabilities of R from M6 and Q from I5. Next add +1 to **each of the 20 standard amino acids** at each emitting state, including unobserved residues in the denominator. Calculate the two probabilities again. Keep the smoothed M6(R) probability for **Q8**.

Transitions and emissions have separate probability distributions. Do not combine their denominators. Delete states emit no amino acid.

In [ ]:
hmm_emission_answers = {
 "M6_counts_K_R_H": None,
 "M6_raw_probability_R": None,
 "I5_counts_Q_N_A": None,
 "I5_raw_probability_Q": None,
 "M6_plus1_probability_R": None,
 "I5_plus1_probability_Q": None,
}

In [ ]:
check("HMM emissions",hmm_emission_answers,{
 "M6_counts_K_R_H": (1,2,2), "M6_raw_probability_R": 2/5,
 "I5_counts_Q_N_A": (1,1,1), "I5_raw_probability_Q": 1/3,
 "M6_plus1_probability_R": 3/25, "I5_plus1_probability_Q": 2/23,
},tol=0.005)

The next cell builds the complete model from the training alignment with the same +1 rules you used. Read the variable names to connect the code to your hand work. Every probability row is checked to sum to one.

In [ ]:
from collections import Counter, defaultdict
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
HMM_AA = "ACDEFGHIKLMNPQRSTVWY"
HMM_MSA = ["MAKLT---K-D", "MAKLT---RP-", "-GKSR---RTE", "MARS-QNAHYN", "MARSR---HWQ"]
HMM_MATCH_COLUMNS = (1,2,3,4,5,9,10,11)
hmm_states = ["B", "I0"] + [letter+str(i) for i in range(1,9) for letter in "MDI"] + ["E"]
hmm_legal = {"B": ["M1","D1","I0"], "I0": ["I0","M1","D1"], "E": []}
for i in range(1,9):
    for letter in "MDI":
        hmm_legal[letter+str(i)] = (["M"+str(i+1),"D"+str(i+1),"I"+str(i)]
                                   if i<8 else ["E","I8"])
hmm_transition_counts = defaultdict(Counter)
hmm_emission_counts = defaultdict(Counter)
hmm_training_paths = []
for row in HMM_MSA:
    state_path = ["B"]
    for column, residue in enumerate(row,1):
        if column in HMM_MATCH_COLUMNS:
            k = HMM_MATCH_COLUMNS.index(column)+1
            state = ("D" if residue=="-" else "M")+str(k)
            state_path.append(state)
            if residue!="-": hmm_emission_counts[state][residue]+=1
        elif residue!="-":
            state_path.append("I5")
            hmm_emission_counts["I5"][residue]+=1
    state_path.append("E")
    hmm_training_paths.append(tuple(state_path))
    for start,end in zip(state_path,state_path[1:]):
        assert end in hmm_legal[start]
        hmm_transition_counts[start][end]+=1
# Separate +1 prior for every declared outgoing edge.
hmm_transition = {state:{target:(hmm_transition_counts[state][target]+1)/
    (sum(hmm_transition_counts[state].values())+len(targets)) for target in targets}
    for state,targets in hmm_legal.items() if targets}
# +1 for ALL 20 amino acids, including unobserved residues.
hmm_emission = {state:{aa:(hmm_emission_counts[state][aa]+1)/
    (sum(hmm_emission_counts[state].values())+20) for aa in HMM_AA}
    for state in hmm_states if state.startswith(("M","I"))}
assert all(np.isclose(sum(row.values()),1) for row in hmm_transition.values())
assert all(np.isclose(sum(row.values()),1) for row in hmm_emission.values())
assert "D5" not in hmm_emission
print("Built the eight-match-state toy model; every probability row sums to 1.")

### 3D. Viterbi calculation and traceback (20 minutes)

Use the lecture's query **MGLTARPQ** with our declared, fully smoothed model. A path probability multiplies transitions and emissions. Working in log base 2 turns products into sums. These are **log probabilities**, rather than background-adjusted bit scores or E-values.

V(state, j) is the highest log probability of a path ending at that state after consuming j query residues. For an emitting M or I state, candidates use predecessors at j−1; add a transition log and an emission log. For a silent D state, use predecessors at the same j and no emission term. Choose the maximum and retain the pointer that produced it.

Run the next cell. The full eight-match-state dynamic-programming calculation is supplied, with three values hidden in a small window. For each missing entry, use the displayed predecessor rows to calculate **max(V predecessor + log transition + log emission)**. The missing entries cover a silent D update, an emitting I update and an emitting M update. A competing path need not lie on the final winner. Enter your values to two decimal places.

Next perform the traceback by hand using the complete pointer grid. Start at **E, j=8**, read its predecessor state and j, and continue until **B, j=0**. Reverse your visited-state list and enter it as a **forward B-to-E path** in hmm_hand_path, using spaces between state names. Then run the check to compare your traceback with the algorithm and use the full path for **Q9**.

In [ ]:
def hmm_viterbi(sequence):
    """Maximum log2 joint probability; silent D states consume no query residue."""
    values={(s,j):-math.inf for s in hmm_states for j in range(len(sequence)+1)}
    previous={}
    values[("B",0)]=0.0
    # Silent D edges move left to right at fixed j, hence this order is acyclic.
    for j in range(len(sequence)+1):
        for state in hmm_states[1:]:
            emits=state.startswith(("M","I"))
            previous_j=j-1 if emits else j
            if previous_j<0: continue
            emission_log=math.log2(hmm_emission[state][sequence[j-1]]) if emits else 0.0
            candidates=[]
            for pred in hmm_states:
                probability=hmm_transition.get(pred,{}).get(state)
                if probability is not None:
                    candidate=values[(pred,previous_j)]+math.log2(probability)+emission_log
                    candidates.append((candidate,pred,previous_j))
            if candidates:
                value,pred,pred_j=max(candidates,key=lambda item:item[0])
                values[(state,j)]=value
                previous[(state,j)]=(pred,pred_j)
    end=("E",len(sequence))
    if not math.isfinite(values[end]): raise ValueError("No legal path emits this query.")
    trace=[end]
    while trace[-1]!=("B",0): trace.append(previous[trace[-1]])
    return values,list(reversed(trace)),previous

HMM_QUERY="MGLTARPQ"
hmm_dp,hmm_trace,hmm_pointers=hmm_viterbi(HMM_QUERY)
hmm_missing={"V_D5_4":("D5",4),"V_I5_5":("I5",5),"V_M6_6":("M6",6)}
window=pd.DataFrame({j:{s:hmm_dp[(s,j)] for s in ("M4","D5","M5","I5","M6")}
                     for j in (3,4,5,6)})
window=window.apply(lambda col:col.map(lambda v:"-inf" if not math.isfinite(v) else f"{v:.3f}"))
for state,j in hmm_missing.values(): window.loc[state,j]="?"
print("j counts consumed query residues. Values are log2 joint probabilities.")
display(window)
pointer_grid=pd.DataFrame({j:{state:("START" if (state,j)==("B",0) else
    (f"{hmm_pointers[(state,j)][0]},{hmm_pointers[(state,j)][1]}"
     if math.isfinite(hmm_dp[(state,j)]) and (state,j) in hmm_pointers else "—"))
    for state in hmm_states} for j in range(len(HMM_QUERY)+1)})
print("Traceback pointers: row=current state, column=current j; entry=previous state,previous j.")
with pd.option_context("display.max_rows",40,"display.max_columns",12):
    display(pointer_grid)
for label,(state,j) in hmm_missing.items():
    emits=state.startswith(("M","I")); prev_j=j-1 if emits else j
    rows=[]
    for pred in hmm_states:
        if state in hmm_transition.get(pred,{}):
            rows.append({"predecessor":f"{pred}, j={prev_j}",
                "V(predecessor)":hmm_dp[(pred,prev_j)],
                "log2 transition":math.log2(hmm_transition[pred][state]),
                "log2 emission":math.log2(hmm_emission[state][HMM_QUERY[j-1]]) if emits else 0.0})
    print(label,"for query residue",HMM_QUERY[j-1])
    display(pd.DataFrame(rows).round(3))

In [ ]:
hmm_viterbi_answers = {
 "V_D5_4": None,
 "V_I5_5": None,
 "V_M6_6": None,
}
hmm_hand_path = None  # space-separated forward path, including B and E

In [ ]:
hmm_viterbi_expected={label:hmm_dp[state_j] for label,state_j in hmm_missing.items()}
check("Viterbi entries",hmm_viterbi_answers,hmm_viterbi_expected,tol=0.02)
hmm_expected_path=" ".join(state for state,_ in hmm_trace)
check("Hand traceback",hmm_hand_path,hmm_expected_path)
hmm_viterbi_ready=all(isinstance(hmm_viterbi_answers[k],(int,float)) and
    math.isclose(hmm_viterbi_answers[k],value,abs_tol=0.02)
    for k,value in hmm_viterbi_expected.items()) and isinstance(hmm_hand_path,str) and " ".join(hmm_hand_path.split())==hmm_expected_path
if hmm_viterbi_ready:
    print("Highest-probability path:"," ".join(state for state,_ in hmm_trace))
    print("log2 joint probability:",round(hmm_dp[("E",len(HMM_QUERY))],6))
    print("Query residue 5 is",HMM_QUERY[4],"and is emitted by",
        next(state for (state,j),(_,prev_j) in zip(hmm_trace[1:],hmm_trace)
             if j==5 and prev_j==4 and state.startswith(("M","I"))))
    fig,ax=plt.subplots(figsize=(10,2.8))
    names=[state for state,_ in hmm_trace]
    x=np.arange(len(names))
    ax.plot(x,np.zeros(len(names)),"o-",color="#136f63",lw=2,ms=9)
    for k,name in enumerate(names): ax.text(k,0.1,name,ha="center")
    ax.set(xlim=(-.5,len(names)-.5),ylim=(-.25,.45),title="Viterbi path for MGLTARPQ; other edges omitted")
    ax.axis("off")
    plt.show()
else:
    print("Finish the three hand calculations and your hand traceback to reveal the path.")

**If you finish early.** Try the alternative nine-residue query **MAKLTARPQ**. It differs from MGLTARPQ by G→A and an added K. Predict where an insertion will align, then run the optional cell. The model stays fixed; only the query changes. Explain why a deletion changes model position without emitting a residue, whereas an insertion can emit several residues at one model position.

In [ ]:
if hmm_viterbi_ready:
    variant_dp,variant_trace,variant_pointers=hmm_viterbi("MAKLTARPQ")
    print("Optional query: MAKLTARPQ")
    print("Path:"," ".join(state for state,_ in variant_trace))
    print("log2 joint probability:",round(variant_dp[("E",9)],6))
else:
    print("Optional extension: finish the core Viterbi entries first.")

## Part 4 — Interpret real globin evidence

The short alignments above made every calculation visible. Now consider real proteins: haemoglobin beta (HBB, UniProt P68871) and myoglobin (MB, P02144). A weak hit from one search does not by itself show that two proteins are unrelated. Inspect what was searched, what alignment was found and what a family model adds.

The first table is the **saved BLAST+ 2.16.0 example from the course teaching notebook**, copied into this lab on 14 November 2026. The original database contained five other human globins (HBD, HBE, HBA, HBAZ and MB); it was not a search of all UniProt. The table below is a stored result, not a search launched by this notebook.

Inspect the two rows and answer on paper:

1. Does E = 63 provide strong evidence for the reported alignment?
2. What changes when the two-hit requirement is removed?
3. What does this comparison tell you about a heuristic missing an alignment?

The actual bit scores and E-values below come from that saved search. They should not be recomputed using the simplified parameters of Part 1C.

In [ ]:
blast_example = pd.DataFrame([
    {'setting': 'Original two-hit setting', 'query': 'HBB (P68871)', 'target': 'MB (P02144)',
     'identity_percent': 16.000, 'alignment_columns': 125, 'E_value': 63.0, 'bits': 10.8},
    {'setting': 'Two-hit requirement disabled', 'query': 'HBB (P68871)', 'target': 'MB (P02144)',
     'identity_percent': 25.517, 'alignment_columns': 145, 'E_value': 5.38e-11, 'bits': 45.4},
])
display(blast_example)
print('Recorded BLAST settings: protein search; reporting E-value cutoff 1000;')
print('the second search adds -window_size 0 and -max_hsps 1.')

### Retrieve family information from a saved annotation snapshot
Before displaying the next table, predict whether a globin-family profile could support a relationship that the original BLAST result did not detect well.

The following data were retrieved from **InterPro version 111.0 on 14 November 2026**. The API supplies precomputed Pfam annotations for known accessions; this is an annotation lookup, not a newly submitted sequence search. The raw responses and their retrieval details are embedded below so the activity works offline.

After running the cell, identify the family accession shared by the proteins and compare the matched coordinates with each full protein length. These are matched regions, not evidence that every residue of both proteins has the same role.

In [ ]:
PFAM_SNAPSHOT = {'P68871': {'response': {'count': 1, 'next': None, 'previous': None, 'results': [{'metadata': {'accession': 'PF00042', 'name': 'Globin', 'source_database': 'pfam', 'type': 'domain', 'integrated': 'IPR000971', 'member_databases': None, 'go_terms': None}, 'proteins': [{'accession': 'p68871', 'protein_length': 147, 'source_database': 'reviewed', 'organism': '9606', 'in_alphafold': True, 'entry_protein_locations': [{'fragments': [{'start': 27, 'end': 142, 'dc-status': 'CONTINUOUS'}], 'representative': False, 'model': 'PF00042', 'score': 1.3e-29}]}]}]}, 'retrieved_at': '2026-11-14T18:05:42.994462+00:00', 'url': 'https://www.ebi.ac.uk/interpro/api/entry/pfam/protein/uniprot/P68871/', 'interpro_version': '111.0'}, 'P02144': {'response': {'count': 1, 'next': None, 'previous': None, 'results': [{'metadata': {'accession': 'PF00042', 'name': 'Globin', 'source_database': 'pfam', 'type': 'domain', 'integrated': 'IPR000971', 'member_databases': None, 'go_terms': None}, 'proteins': [{'accession': 'p02144', 'protein_length': 154, 'source_database': 'reviewed', 'organism': '9606', 'in_alphafold': True, 'entry_protein_locations': [{'fragments': [{'start': 28, 'end': 141, 'dc-status': 'CONTINUOUS'}], 'representative': False, 'model': 'PF00042', 'score': 1.2e-20}]}]}]}, 'retrieved_at': '2026-11-14T18:05:43.560397+00:00', 'url': 'https://www.ebi.ac.uk/interpro/api/entry/pfam/protein/uniprot/P02144/', 'interpro_version': '111.0'}}

family_rows=[]
for acc, record in PFAM_SNAPSHOT.items():
    payload=record['response']
    if payload.get('next') is not None:
        raise ValueError('The saved snapshot must include all pages.')
    for entry in payload['results']:
        for protein in entry['proteins']:
            for occurrence, location in enumerate(protein['entry_protein_locations'], 1):
                for fragment, region in enumerate(location['fragments'], 1):
                    family_rows.append({
                        'protein': 'HBB' if acc == 'P68871' else 'MB',
                        'UniProt': acc, 'family': entry['metadata']['accession'],
                        'name': entry['metadata']['name'], 'occurrence': occurrence,
                        'fragment': fragment, 'start': region['start'], 'end': region['end'],
                        'protein_length': protein['protein_length'],
                        'E_value': float(location['score'])})
family_matches = pd.DataFrame(family_rows)
family_matches['fragment_coverage_percent'] = 100 * (
    family_matches['end'] - family_matches['start'] + 1) / family_matches['protein_length']
display(family_matches)
fig, ax=plt.subplots(figsize=(9, 2.8))
for y, (_, row) in enumerate(family_matches.iterrows()):
    ax.plot([1, row.protein_length], [y,y], color='#c4cbd0', linewidth=10, solid_capstyle='butt')
    ax.plot([row.start, row.end], [y,y], color='#27758c', linewidth=10, solid_capstyle='butt')
    ax.text(row.start, y+.12, f'{row.family}: {row.start}–{row.end}', fontsize=9)
ax.set_yticks(range(len(family_matches)), family_matches['protein'])
ax.set(xlabel='Residue position (1-based)', title='Saved Pfam Globin matches', ylim=(-.5,1.6))
fig.tight_layout(); plt.show()

### Explain what the evidence supports — Canvas Q10
Write three sentences in your notes:

- Explain why the initial weak BLAST alignment does not prove that HBB and MB are unrelated.
- State what their shared Pfam family match supports, keeping the conclusion within the matched region.
- Explain why neither the strongest match nor the smallest E-value across these different search models establishes identical biological function.

**One additional modelling question:** suppose a profile-training alignment contains 50 nearly identical mammalian sequences and three distant relatives. What could go wrong if every sequence contributed equally? Explain how down-weighting near-duplicates can help the model describe the family rather than its most heavily sampled subgroup.

The two comparisons address different questions: changing BLAST's heuristic shows an alignment can be missed; a family-profile annotation contributes independent family evidence. The data are not a controlled ranking of BLAST versus HMMER sensitivity.

## Review and submit

Check your Canvas responses against these notebook activities:

| Question | Evidence to use |
|---|---|
| Q1 | LWG/IWG word-score calculation |
| Q2 | Peak, trim endpoint and stopping column in the X-drop trace |
| Q3 | Database-size calculation and bit-score prediction |
| Q4 | Smoothed G score in PSSM position 1 |
| Q5 | Difference between AGLSP and GGLSP scores |
| Q6 | Best placement of one gap in RASRP |
| Q7 | Unsmoothed I5 self-transition estimate |
| Q8 | Smoothed arginine emission probability at M6 |
| Q9 | Viterbi path for MGLTARPQ under the supplied model |
| Q10 | Interpretation of the real BLAST and Pfam evidence |

Keep your working and explanations in this notebook. Save your notebook, submit the ten-question quiz in Canvas and complete the Session 5 lab feedback survey in the same module. See the Canvas module for the deadline and course-wide submission rules.

## Optional extension after the required lab
Change one parameter at a time and record a prediction before running: the BLAST word threshold, X-drop threshold, profile pseudocount strength or HMM query. Explain which output changes and why. For the HMM, compare the two supplied query sequences and identify the insertion state in the second path.

If you rerun a real search independently, record the exact sequences, database/release, settings, date and result. Treat changing live results as observations to explain, rather than expecting the saved numbers to recur.

## Data and model references
- Worked alignments and sentence example: Petras Kundrotas, Advanced Database Search, February 2027 lecture, slides 6–121. Calculations here follow the explicit model rules given in each activity.
- BLOSUM62 substitution scores: embedded standard 20-amino-acid table, read from Biopython 1.88's BLOSUM62 data during preparation; no Biopython installation is required to run this lab. Original matrix: Henikoff & Henikoff (1992), [PNAS 89:10915–10919](https://doi.org/10.1073/pnas.89.22.10915).
- Full BLOSUM62 background frequencies: [HMMER/Easel composition source](https://github.com/EddyRivasLab/easel/blob/master/esl_composition.c), function esl_composition_BL62. The twenty rounded values sum to one.
- [NCBI alignment-score statistics](https://blast.ncbi.nlm.nih.gov/tutorial/Altschul-1.html): bit scores, E-values and search space.
- Saved BLAST transcript: Session 5 course teaching notebook, BLAST+ 2.16.0, recorded commands and five-target database. The original experiment date is not supplied with that transcript; 14 November 2026 is the date this lab copied it.
- [HBB Pfam annotations](https://www.ebi.ac.uk/interpro/api/entry/pfam/protein/uniprot/P68871/) and [myoglobin Pfam annotations](https://www.ebi.ac.uk/interpro/api/entry/pfam/protein/uniprot/P02144/): retrieved 14 November 2026, InterPro version 111.0; raw data and provenance are embedded in the data cell. [Pfam API documentation](https://pfam-docs.readthedocs.io/en/latest/api.html) explains annotation retrieval.